# Decision trees for spam and bike rentals

Learn the split calculation on measured email features, then switch to forecasting measured hourly bike rentals. UCI Spambase records the longest consecutive uppercase run (X), the frequency of the word **free**, and a label: Legitimate or Spam. We derive Y by testing whether that frequency is positive. The eight training emails and six evaluation emails below are deliberately selected to support hand calculations.

**Live route.** Run all cells, then start screen sharing at **Score the first cut**. Change one parameter and run that cell again. Reveal the fitted tree and depth comparison. Every rendered tree keeps the clicked split’s counts, weighted score, and gain beside the visual.

[UCI Spambase](https://archive.ics.uci.edu/dataset/94/spambase) · [UCI Bike Sharing](https://archive.ics.uci.edu/dataset/275/bike-sharing-dataset) · [Interactive lesson](https://sharanharsoor.github.io/ml-learning-lab/decision-trees/)

In [ ]:
from collections import Counter
from html import escape
from math import isclose, log2
from statistics import mean

from IPython.display import HTML, display
from sklearn import __version__ as sklearn_version
from sklearn.metrics import confusion_matrix
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

# One-based line, uppercase run X, UCI percentage for the word free, observed class
train_source = [(1816, 1, 0.0, 0),
 (2501, 2, 3.7, 0),
 (2091, 3, 0.68, 0),
 (7, 4, 0.96, 1),
 (24, 5, 1.16, 1),
 (311, 6, 0.43, 1),
 (1817, 7, 0.0, 0),
 (48, 8, 4.97, 1)]
held_out_source = [(1815, 2, 0.0, 0),
 (2689, 3, 1.63, 0),
 (191, 5, 2.25, 1),
 (63, 8, 1.39, 1),
 (11, 6, 0.0, 1),
 (2237, 7, 1.85, 0)]


def classification_inputs(records):
    return [(x, 'B' if free else 'A', label) for _, x, free, label in records]


train = classification_inputs(train_source)
held_out = classification_inputs(held_out_source)
routes = ('Legitimate', 'Spam')
print(f'{len(train)} training emails · {len(held_out)} evaluation emails · scikit-learn {sklearn_version}')


def records_table(caption, headers, rows):
    heading = ''.join(f'<th scope="col">{escape(str(value))}</th>' for value in headers)
    body = ''.join(
        '<tr>' + ''.join(f'<td>{escape(str(value))}</td>' for value in row) + '</tr>'
        for row in rows
    )
    return (f'<div style="overflow-x:auto"><table><caption>{escape(caption)}</caption>'
            f'<thead><tr>{heading}</tr></thead><tbody>{body}</tbody></table></div>')


training_records = [
    (f'E{i}', line, x, 'B' if free else 'A', routes[label])
    for i, (line, x, free, label) in enumerate(train_source, 1)
]
display(HTML(records_table('Selected UCI Spambase training emails',
    ('Email', 'UCI source line', 'Capital run X', 'Free Y', 'Observed label'), training_records)))

## Score the first cut

The root has four Legitimate and four Spam emails. Its entropy is **1 bit**, and its Gini impurity is **0.5**. Entropy averages surprise. Gini measures the probability that two draws with replacement disagree.

At **X ≤ 3.5**, E1–E3 go Yes and E4–E8 go No. Entropy of the Yes branch is zero. The No branch has probabilities 1/5 and 4/5, so its entropy is −[(1/5) log₂(1/5) + (4/5) log₂(4/5)] ≈ 0.7219. Multiply by the child row fractions: (3/8)(0) + (5/8)(0.7219) ≈ 0.4512. Information gain is 1 − 0.4512 ≈ 0.5488 bits.

Change `threshold` from 3.5 to 2.5, or `criterion` from `entropy` to `gini`. Run the cell again and read both groups before comparing the results.

In [ ]:
def counts_score(counts, criterion):
    total = sum(counts)
    p = [count / total for count in counts if count] if total else []
    if not p:
        return 0.0
    if criterion == 'entropy':
        return sum(-q * log2(q) for q in p)
    if criterion == 'gini':
        return 1 - sum(q ** 2 for q in p)
    raise ValueError('Choose entropy or gini.')


def split_score(children, criterion):
    parent = [sum(values) for values in zip(*children)]
    n = sum(parent)
    before = counts_score(parent, criterion)
    scores = [counts_score(counts, criterion) for counts in children]
    after = sum(sum(counts) / n * score for counts, score in zip(children, scores))
    return before, scores, after, before - after


def binary_counts(rows, goes_left):
    left = Counter(row[2] for row in rows if goes_left(row))
    right = Counter(row[2] for row in rows if not goes_left(row))
    return [[group[0], group[1]] for group in (left, right)]


fixed_candidates = {
    'Capital run ≤ 3.5': binary_counts(train, lambda row: row[0] <= 3.5),
    'Free absent': binary_counts(train, lambda row: row[1] == 'A'),
}

for question, children in fixed_candidates.items():
    before, scores, after, gain = split_score(children, 'entropy')
    print(f'{question}   weighted child entropy {after:.4f} bits   gain {gain:.4f} bits')

assert isclose(split_score(fixed_candidates['Capital run ≤ 3.5'], 'entropy')[3], .5487949406953986)
assert isclose(split_score(fixed_candidates['Capital run ≤ 3.5'], 'gini')[3], .3)

In [ ]:
threshold = 3.5
criterion = 'entropy'
children = binary_counts(train, lambda row: row[0] <= threshold)
if not all(map(sum, children)):
    print('Choose a threshold that leaves at least one email on each side.')
else:
    before, scores, after, gain = split_score(children, criterion)
    for name, counts, score in zip(('Yes', 'No'), children, scores):
        print(f'{name}   {counts[0]} Legitimate + {counts[1]} Spam   {criterion} {score:.4f}')
    print(f'{before:.4f} parent − {after:.4f} weighted children = {gain:.4f} gain')

In [ ]:
theme = """
<style>
.dt-out{font:16px/1.48 system-ui;color:#193d38;border:1px solid #a7c6b9;border-radius:14px;
  background:#f6fbf8;padding:16px;margin:10px 0;max-width:980px}
.dt-out h3{font-size:1.35em;margin:0 0 .45em}
.dt-out h4{font-size:1.05em;margin:0 0 .45em}
.dt-out p{margin:.5em 0}
.dt-out details{border-block-start:1px solid #c7dbd1;margin-top:.7em;padding:.45em 0}
.dt-out summary{cursor:pointer;font-weight:700}
.dt-out table{border-collapse:collapse;text-align:left;max-width:100%;margin:.45em 0}
.dt-out :is(th,td){padding:.32em .6em;border-bottom:1px solid #d6e7dc}
.dt-out .scroll{overflow-x:auto}
.dt-out .band{display:flex;height:11px;border-radius:5px;overflow:hidden;margin:.18em 0 .35em}
.dt-out .branches{display:grid;grid-template-columns:repeat(auto-fit,minmax(min(100%,175px),1fr));gap:.6em}
.dt-out .branch{background:white;border:1px solid #c7dbd1;border-radius:8px;padding:.65em}
.dt-out svg{display:block;width:100%;min-width:560px}
.dt-out .crumb{font-size:.9em;color:#4d655d}
</style>
"""


def panel(title, content):
    display(HTML(theme + f'<section class="dt-out" data-visual="decision-trees">'
                f'<h3>{escape(title)}</h3>{content}</section>'))


def reveal(title, body, opened=False):
    flag = ' open' if opened else ''
    return f'<details{flag}><summary>{escape(title)}</summary>{body}</details>'


def split_cards(children, names, outcomes, criterion):
    before, scores, after, gain = split_score(children, criterion)
    total = sum(map(sum, children))
    palette = ('#087669', '#ae5c3b', '#64479e')
    cards = []
    for name, counts, score in zip(names, children, scores):
        n = sum(counts)
        bars = ''.join(f'<span style="width:{count / n * 100}%;background:{color}"></span>'
                       for count, color in zip(counts, palette))
        labels = ' · '.join(f'{count} {label}' for count, label in zip(counts, outcomes))
        cards.append(f'<article class="branch"><h4>{escape(name)}</h4><p>{n}/{total} of parent</p>'
                     f'<div class="band" aria-hidden="true">{bars}</div><p>{labels}</p>'
                     f'<p>{criterion.capitalize()} {score:.4f}</p></article>')
    terms = ' + '.join(f'({sum(counts)}/{total}) × {score:.4f}'
                       for counts, score in zip(children, scores))
    return (f'<div class="branches">{"".join(cards)}</div>'
            f'<p>Weighted child score = {terms} ≈ <strong>{after:.4f}</strong></p>'
            f'<p>Gain = {before:.4f} − {after:.4f} ≈ <strong>{gain:.4f}</strong></p>')


question_panels = []
for name, children in fixed_candidates.items():
    for criterion in ('entropy', 'gini'):
        question_panels.append(reveal(
            f'{name} · {criterion.capitalize()}',
            split_cards(children, ('Yes child', 'No child'), routes, criterion),
            opened=(name == 'Capital run ≤ 3.5' and criterion == 'entropy'),
        ))
panel('Compare the first questions', ''.join(question_panels))

### Which numeric cutoffs get tested?

Sort distinct uppercase-run lengths and check one midpoint per gap. For integers 3 and 4, a threshold of 3.5 separates those training rows. The simplified coarse search tests the edges 2.5, 4.5, and 6.5. The free indicator competes in both modes.

The optional invented extension illustrates a different distinction: the sender feature has three categories (known, first-time, unknown), while the target has three labels (Legitimate, Spam, Suspicious). Two branches can still predict any of the three labels.

In [ ]:
values = sorted({row[0] for row in train})
midpoints = [(a + b) / 2 for a, b in zip(values, values[1:])]
coarse_edges = [2.5, 4.5, 6.5]

for mode, thresholds in [('Exact midpoints', midpoints), ('Coarse edges', coarse_edges)]:
    scored = [
        (cut, split_score(binary_counts(train, lambda row: row[0] <= cut), 'entropy')[3])
        for cut in thresholds
    ]
    best_cut, best_gain = max(scored, key=lambda item: item[1])
    print(f'{mode}   best numeric cutoff {best_cut:g}   gain {best_gain:.4f} bits')

a, b, c = [3, 1, 0], [0, 3, 1], [0, 0, 4]
outcomes = ('Legitimate', 'Spam', 'Suspicious')
options = [
    ('Known sender or other', [a, [x + y for x, y in zip(b, c)]],
     ('Known sender', 'First-time sender or C')),
    ('One branch per sender group', [a, b, c], ('Known sender', 'First-time sender', 'Unknown sender')),
]
panel('Two or three children', ''.join(
    reveal(name, split_cards(groups, children, outcomes, 'entropy'), opened=i == 0)
    for i, (name, groups, children) in enumerate(options)
))

## Fit a tree and trace an email

The new email has the same inputs as E7, a capital run of 7 and the word **free** absent. E7’s real label is Legitimate. For the new email we only supply the two inputs. The root tests 7 ≤ 3.5 (No). The next question asks whether free is absent (Yes). The leaf predicts Legitimate based on the one training example that reached it.

Every descendant scores questions using the rows it received. Row fractions in a split’s weighted score always use its immediate parent.

In [ ]:
def input_columns(rows):
    return [[row[0], int(row[1] == 'B')] for row in rows]


def fit_classifier(depth, min_leaf=1, alpha=0):
    return DecisionTreeClassifier(
        criterion='entropy', max_depth=depth, min_samples_leaf=min_leaf,
        ccp_alpha=alpha, random_state=0,
    ).fit(input_columns(train), [row[2] for row in train])


classifier = fit_classifier(2)
email = [7, 0]
assert classifier.predict([email]).tolist() == [0]

In [ ]:
def tree_picture(model, features, example, numerical=False):
    tree = model.tree_
    on_path = set(model.decision_path([example]).indices)
    xy = {}
    edges = []
    split_notes = []
    next_slot = 0
    unit = ' rentals' if numerical else ''

    def position(node, depth):
        nonlocal next_slot
        left, right = tree.children_left[node], tree.children_right[node]
        y = 44 + depth * 112
        count = tree.n_node_samples[node]
        if left < 0:
            x = 72 + 147 * next_slot
            next_slot += 1
            value = tree.value[node][0]
            first = f'{value[0]:g}{unit}' if numerical else routes[model.classes_[value.argmax()]]
            xy[node] = (x, y, first, f'{count} training rows')
            return x

        x = (position(left, depth + 1) + position(right, depth + 1)) / 2
        feature = features[tree.feature[node]]
        threshold = tree.threshold[node]
        question = 'Free absent?' if feature == 'Free present' and isclose(threshold, .5) else f'{feature} ≤ {threshold:g}?'
        xy[node] = (x, y, question, f'{count} training rows')
        for side, child in [('Yes', left), ('No', right)]:
            cx, cy, *_ = xy[child]
            highlighted = child in on_path
            color = '#64479e' if highlighted else '#94b2a7'
            edges.append(f'<path d="M{x} {y+24} L{cx} {cy-24}" stroke="{color}" '
                         f'stroke-width="{4 if highlighted else 2}" />'
                         f'<text x="{(x+cx)/2}" y="{(y+cy)/2-5}" '
                         f'font-size="12" text-anchor="middle" '
                         f'paint-order="stroke" stroke="#f6fbf8" stroke-width="4">{side}</text>')
        wl, wr = tree.weighted_n_node_samples[[left, right]]
        total = wl + wr
        il, ir = tree.impurity[[left, right]]
        weighted = wl / total * il + wr / total * ir
        gain = tree.impurity[node] - weighted
        split_notes.append((depth, node, reveal(
            f'Depth {depth} · {question}',
            (f'<p>Child means {tree.value[left][0, 0]:.3f} and {tree.value[right][0, 0]:.3f} rentals.</p>' if numerical else '') +
            f'<p>Parent {total:g} rows · child scores {il:.4f} and {ir:.4f}</p>'
            f'<p>Weighted child score = ({wl:g}/{total:g}) × {il:.4f} + '
            f'({wr:g}/{total:g}) × {ir:.4f} ≈ <strong>{weighted:.4f}</strong>.</p>'
            f'<p>Gain = {tree.impurity[node]:.4f} − {weighted:.4f} ≈ {gain:.4f}. '
            f'Across all {tree.n_node_samples[0]} training rows, '
            f'credit = ({total:g}/{tree.n_node_samples[0]}) × {gain:.4f} '
            f'≈ {total / tree.n_node_samples[0] * gain:.4f}.</p>',
        )))
        return x

    position(0, 0)
    nodes = []
    for node, (x, y, first, second) in xy.items():
        active = node in on_path
        fill = '#eee6fc' if active else 'white'
        border = '#64479e' if active else '#a4c2b5'
        nodes.append(f'<g><rect x="{x-63}" y="{y-24}" width="126" height="48" fill="{fill}" '
                     f'stroke="{border}" stroke-width="{3 if active else 1}" rx="8" />'
                     f'<text x="{x}" y="{y-3}" text-anchor="middle" font-size="12" font-weight="700">{escape(first)}</text>'
                     f'<text x="{x}" y="{y+13}" text-anchor="middle" font-size="11">{escape(second)}</text></g>')
    height = 90 + tree.max_depth * 112
    width = max(294, next_slot * 147)
    diagram = (f'<div class="scroll"><svg viewBox="0 0 {width} {height}" '
               f'role="img" aria-label="Fitted tree. The purple path traces the input record.">'
               f'{"".join(edges)}{"".join(nodes)}</svg></div>')
    result = model.predict([example])[0]
    prediction = f'{result:g} rentals' if numerical else routes[result]
    return (f'<p><strong>Prediction {prediction}</strong> · purple shows the path for this record.</p>'
            + diagram + ''.join(note for _, _, note in sorted(split_notes)))


panel('Trace a capital run of 7 with free absent', tree_picture(classifier, ('Capital run', 'Free present'), email))

### Read the immediate parent further down

This separate selection contains **16 measured emails**, all with free absent. At depth 3, the selected parent has seven rows. Three go Yes with zero entropy and four go No with one bit. Their weighted entropy is (3/7)(0) + (4/7)(1) = 0.5714 bits. The denominator 16 is for tree-wide feature credit only.

In [ ]:
# Source line, capital-run length, and observed class
selected_deep = [(1821, 1, 0),
 (1839, 2, 0),
 (68, 3, 1),
 (19, 4, 1),
 (34, 5, 1),
 (1880, 6, 0),
 (1822, 7, 0),
 (58, 8, 1),
 (21, 9, 1),
 (1843, 10, 0),
 (1819, 11, 0),
 (1825, 12, 0),
 (1952, 13, 0),
 (594, 14, 1),
 (6, 15, 1),
 (77, 16, 1)]
route_batch = [label for _, _, label in selected_deep]
route_tree = DecisionTreeClassifier(criterion='entropy', max_depth=5, random_state=0).fit(
    [[capital_run] for capital_run in range(1, 17)], route_batch,
)
trace = route_tree.decision_path([[7]]).indices
parent_node = trace[3]
tree = route_tree.tree_
capital_runs = list(range(1, 17))
membership = route_tree.decision_path([[capital_run] for capital_run in capital_runs]).toarray()
immediate_records = [
    (answer, f'D{capital_run}', selected_deep[i][0], capital_run, routes[route_batch[capital_run - 1]])
    for answer, child in [('Yes', tree.children_left[parent_node]), ('No', tree.children_right[parent_node])]
    for i, capital_run in enumerate(capital_runs) if membership[i, child]
]
panel('Which rows count at depth 3?', tree_picture(route_tree, ('Capital run',), [7]) + records_table(
    'Only these seven rows enter the depth 3 calculation',
    ('Branch', 'Email', 'UCI source line', 'Capital run X', 'Observed label'), immediate_records,
))
assert Counter(row[0] for row in immediate_records) == {'Yes': 3, 'No': 4}
assert route_tree.get_depth() == 5

## Choose where to stop

Depth 1 classifies seven of eight training emails and five of six unseen examples correctly. Depth 2 fits eight of eight in training but gets four of six unseen examples. Compare both predictions for H5. The training leaf that changed its prediction was pure because it contained one email.

The six emails illustrate the risk of overfitting. To choose depth in a model study, tune using validation data and measure the final selection on a separate test set.

In [ ]:
def depth_case(depth, leaf_size=1):
    model = fit_classifier(depth, leaf_size)
    train_score = model.score(input_columns(train), [row[2] for row in train])
    y = [row[2] for row in held_out]
    predictions = model.predict(input_columns(held_out))
    matrix = confusion_matrix(y, predictions, labels=[0, 1])
    tn, fp, fn, tp = matrix.ravel()
    table = ('<table><caption>Actual rows, predicted columns</caption>'
             '<thead><tr><th></th><th>Legitimate</th><th>Spam</th></tr></thead>'
             f'<tbody><tr><th>Legitimate</th><td>TN {tn}</td><td>FP {fp}</td></tr>'
             f'<tr><th>Spam</th><td>FN {fn}</td><td>TP {tp}</td></tr></tbody></table>')
    records = [(f'H{i}', x, free_flag, routes[actual], routes[prediction])
               for i, ((x, free_flag, actual), prediction) in enumerate(zip(held_out, predictions), 1)]
    rows = records_table('Six held-out email predictions',
                         ('Email', 'Capital run X', 'Free Y', 'Observed label', 'Prediction'), records)
    return f'<p>Training {train_score:.1%} · held-out {(tn + tp) / len(y):.1%}</p>' + table + rows


panel('One question or two?', reveal('Depth 1 · one question', depth_case(1), opened=True)
      + reveal('Depth 2 · up to two questions', depth_case(2))
      + reveal('Depth 2 · minimum two rows per leaf', depth_case(2, 2)))
assert confusion_matrix([row[2] for row in held_out],
                        classifier.predict(input_columns(held_out)), labels=[0, 1]).tolist() == [[2, 1], [1, 2]]
print('Leaf counts by pruning penalty', {a: int(fit_classifier(2, alpha=a).get_n_leaves()) for a in (.2, .47)})

## Give the deeper tree’s features credit

This selection uses 20 measured emails. Z buckets the percentage of exclamation characters: 2 for zero, 3 for more than 0% through 0.2%, and 4 for more than 0.2%. Local gain uses the immediate parent’s rows. A node containing 13 of 20 training rows and gaining about 0.3600 bits contributes (13/20) × 0.3600 ≈ 0.2340 bits to its chosen feature.

Increase depth from four to five. The exclamation split increases the total credited gain. Capital-run length’s credit can stay the same while its percentage of the total falls.

In [ ]:
# UCI source line, capital run, free %, exclamation %, observed class
importance_records = [(365, 1, 0.0, 0.0, 1),
 (177, 2, 0.0, 0.0, 1),
 (691, 3, 1.5, 0.0, 1),
 (1830, 4, 0.0, 0.546, 0),
 (1828, 5, 0.0, 0.0, 0),
 (3868, 6, 1.75, 0.361, 0),
 (2546, 7, 0.16, 0.0, 0),
 (878, 8, 2.94, 0.0, 1),
 (1882, 9, 0.0, 0.0, 0),
 (225, 10, 0.32, 0.0, 1),
 (12, 11, 0.0, 0.663, 1),
 (1845, 12, 0.0, 0.0, 0),
 (129, 13, 0.0, 0.706, 1),
 (410, 14, 0.25, 0.354, 1),
 (959, 15, 1.21, 0.0, 1),
 (1838, 16, 0.0, 0.0, 0),
 (54, 17, 0.0, 0.198, 1),
 (176, 18, 0.0, 0.968, 1),
 (706, 19, 0.0, 0.0, 1),
 (461, 20, 0.0, 0.176, 1)]
deep_inputs = [[x, int(free > 0), 2 + int(exclaim > 0) + int(exclaim > .2)]
               for _, x, free, exclaim, _ in importance_records]
deep_labels = [label for *_, label in importance_records]
deep_features = ('Capital run', 'Free present', 'Exclaim bucket')
deep_classifiers = {
    depth: DecisionTreeClassifier(criterion='entropy', max_depth=depth, random_state=0)
    .fit(deep_inputs, deep_labels) for depth in (4, 5)
}
reference_email = [11.5, 0, 4]


def credit_table(model):
    tree = model.tree_
    credits = [0.0] * len(deep_features)
    root = tree.weighted_n_node_samples[0]
    for node, feature in enumerate(tree.feature):
        if feature < 0:
            continue
        left, right = tree.children_left[node], tree.children_right[node]
        counts = tree.weighted_n_node_samples
        impurity = tree.impurity
        credit = (counts[node] * impurity[node] - counts[left] * impurity[left]
                  - counts[right] * impurity[right]) / root
        credits[feature] += credit
    rows = ''.join(f'<tr><th>{feature}</th><td>{credit:.4f}</td><td>{share:.2%}</td></tr>'
                   for feature, credit, share in zip(deep_features, credits, model.feature_importances_))
    assert all(isclose(credit / sum(credits), share, abs_tol=1e-10)
               for credit, share in zip(credits, model.feature_importances_))
    return ('<table><thead><tr><th>Feature</th><th>Raw credit in bits</th>'
            f'<th>Share of total</th></tr></thead><tbody>{rows}</tbody></table>')


panel('Which splits earn credit?', ''.join(
    reveal(f'Maximum depth {depth}', credit_table(model) +
           tree_picture(model, deep_features, reference_email), opened=depth == 4)
    for depth, model in deep_classifiers.items()
))

## Predict hourly bike rentals

At the bike-share desk, hourly counts are recorded after the hour. The eight rows used first are measured UCI Capital Bikeshare hours selected for arithmetic practice. At hour ≤ 3.5, the counts on the Yes branch are 2, 2, and 2, so its leaf mean is 2. On the No branch, (6 + 8 + 8 + 4 + 8)/5 = 6.8. Child MSE is 0 and 2.56 rentals². Weighted MSE is (3/8)(0) + (5/8)(2.56) = 1.6 rentals², a reduction of 5.4 from the parent’s 7.

The deeper model fits 20 measured January 2011 rows, one per hour from 04:00 to 23:00. Use hour, working-day category, and relative humidity bucket. During this retrospective exercise the recorded humidity stands in for the forecast reading that would be available before prediction. Trace a February 1 record held out from fitting. Its actual count is 51.

In [ ]:
# instant ID, observed date, hour, count
small_bike_rows = [(49, '2011-01-03', 1, 2),
 (95, '2011-01-05', 2, 2),
 (376, '2011-01-17', 3, 2),
 (1930, '2011-03-26', 4, 6),
 (306, '2011-01-14', 5, 8),
 (576, '2011-01-26', 6, 8),
 (789, '2011-02-05', 7, 4),
 (9, '2011-01-01', 8, 8)]
display(HTML(records_table('Eight selected measured hours', ('UCI ID', 'Date', 'Hour', 'Rentals'), small_bike_rows)))
counts = [count for _, _, _, count in small_bike_rows]
small_regressor = DecisionTreeRegressor(max_depth=1, random_state=0).fit([[row[2]] for row in small_bike_rows], counts)
low_mean, high_mean = small_regressor.predict([[1], [5]])


def mse_at_mean(group):
    center = mean(group)
    return mean((target - center) ** 2 for target in group)


left, right = counts[:3], counts[3:]
weighted_mse = (len(left) * mse_at_mean(left) + len(right) * mse_at_mean(right)) / len(counts)
assert isclose(weighted_mse, 1.6)
assert isclose(low_mean, 2) and isclose(high_mean, 6.8)
print(f'Child means {low_mean:g} and {high_mean:g} rentals · weighted MSE {weighted_mse:g} rentals²')
print(f'Root MSE {mse_at_mean(counts):g} rentals² · reduction {mse_at_mean(counts) - weighted_mse:g} rentals²')

# instant ID, observed date, hour, day type, humidity %, rentals
january_hours = [(529, '2011-01-24', 4, 'A', 48, 1),
 (51, '2011-01-03', 5, 'A', 47, 3),
 (307, '2011-01-14', 6, 'A', 54, 17),
 (356, '2011-01-16', 7, 'B', 56, 3),
 (440, '2011-01-20', 8, 'A', 55, 249),
 (10, '2011-01-01', 9, 'B', 76, 14),
 (148, '2011-01-07', 10, 'A', 37, 63),
 (336, '2011-01-15', 11, 'B', 55, 72),
 (607, '2011-01-28', 12, 'A', 93, 29),
 (362, '2011-01-16', 13, 'B', 35, 118),
 (609, '2011-01-28', 14, 'A', 80, 38),
 (201, '2011-01-09', 15, 'B', 35, 82),
 (472, '2011-01-21', 16, 'A', 26, 97),
 (587, '2011-01-26', 17, 'A', 93, 26),
 (133, '2011-01-06', 18, 'A', 51, 163),
 (43, '2011-01-02', 19, 'B', 42, 30),
 (393, '2011-01-17', 20, 'B', 80, 33),
 (370, '2011-01-16', 21, 'B', 51, 28),
 (254, '2011-01-11', 22, 'A', 93, 29),
 (69, '2011-01-03', 23, 'A', 55, 12)]
display(HTML(records_table('Selected January training hours', ('UCI ID', 'Date', 'Hour', 'Day type', 'Humidity %', 'Rentals'), january_hours)))
bike_features = ('Hour', 'Day type B', 'Humidity bucket')
bike_inputs = [[hour, int(day_type == 'B'), 2 + int(humidity > 60) + int(humidity > 80)]
               for _, _, hour, day_type, humidity, _ in january_hours]
bike_targets = [row[-1] for row in january_hours]
february_example = [11, 0, 4]
actual_count = 51

deep_regressors = {depth: DecisionTreeRegressor(max_depth=depth, random_state=0).fit(bike_inputs, bike_targets)
                   for depth in (4, 5)}
for depth, model in deep_regressors.items():
    predicted = model.predict([february_example])[0]
    print(f'Maximum depth {depth} · prediction {predicted:g} rentals · actual − predicted {actual_count - predicted:g} rentals')

panel('Trace the held-out hour (UCI record 699)', ''.join(
    reveal(f'Maximum depth {depth}', tree_picture(model, bike_features, february_example, numerical=True), opened=depth == 5)
    for depth, model in deep_regressors.items()
))

## Evaluate on more measured hours

The two demonstrations deliberately choose small samples, so their counts and errors describe those examples. For a larger check, run the cell below with `RUN_BENCHMARK = True`. It downloads the official UCI archive, or opens a local copy if you supply its filename. Train on dates before November 1, 2012 and evaluate on the later dates. Compare a lookup trained on hour and working-day group with a depth-five tree trained on the same two features. The chronological split respects the ordering of a forecast. All model inputs are known from the calendar. The observed rental count is only the target. We calculate errors using unrounded predictions. The Regression Metrics lesson scores whole-rental lookup forecasts, with MAE 63.516 and RMSE 98.038 on the same later hours.

[UCI Bike Sharing, Hadi Fanaee-T, CC BY 4.0](https://archive.ics.uci.edu/dataset/275/bike-sharing-dataset)

In [ ]:
from collections import defaultdict
from csv import DictReader
from io import BytesIO, TextIOWrapper
from math import sqrt
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

RUN_BENCHMARK = True
LOCAL_BIKE_ZIP = globals().get('LOCAL_BIKE_ZIP')
UCI_BIKE_ZIP = 'https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip'


def measured_bike_benchmark(local_path=None):
    if local_path:
        archive_bytes = Path(local_path).read_bytes()
    else:
        with urlopen(UCI_BIKE_ZIP, timeout=12) as response:
            archive_bytes = response.read()
    with ZipFile(BytesIO(archive_bytes)) as archive:
        with archive.open('hour.csv') as stored_rows:
            measured = list(DictReader(TextIOWrapper(stored_rows)))
    cutoff = '2012-11-01'
    training = [row for row in measured if row['dteday'] < cutoff]
    testing = [row for row in measured if row['dteday'] >= cutoff]
    assert max(row['dteday'] for row in training) < min(row['dteday'] for row in testing)

    def features(row):
        return int(row['hr']), int(row['workingday'])

    lookup = defaultdict(list)
    training_counts = [int(row['cnt']) for row in training]
    for row, count in zip(training, training_counts):
        lookup[features(row)].append(count)
    group_means = {group: mean(counts) for group, counts in lookup.items()}
    fallback = mean(training_counts)
    benchmark_tree = DecisionTreeRegressor(max_depth=5, random_state=0)
    benchmark_tree.fit([features(row) for row in training], training_counts)
    measured_counts = [int(row['cnt']) for row in testing]

    def errors(predictions):
        residuals = [observed - predicted for observed, predicted in zip(measured_counts, predictions)]
        return f'{mean(abs(value) for value in residuals):.2f}', f'{sqrt(mean(value**2 for value in residuals)):.2f}'

    print(f'{len(training):,} earlier hours fitted; {len(testing):,} later hours evaluated')
    return [
        ('Hour and workday lookup', *errors(group_means.get(features(row), fallback) for row in testing)),
        ('Depth-five tree', *errors(benchmark_tree.predict([features(row) for row in testing]))),
    ]


if RUN_BENCHMARK:
    try:
        scores = measured_bike_benchmark(LOCAL_BIKE_ZIP)
    except OSError:
        print('UCI archive unavailable in this runtime. Set LOCAL_BIKE_ZIP to a downloaded copy and rerun.')
    else:
        display(HTML(records_table('Chronological evaluation on measured UCI hours',
                                   ('Model', 'MAE (rentals)', 'RMSE (rentals)'), scores)))

## Connect the results

Split search compares questions using the current node. Validation checks model choices on fresh records. Feature importance adds improvements across the fitted tree. A class target needs a measure such as entropy or Gini and a class prediction. A numerical target can use MSE and the mean of the destination leaf.

[Continue to Bagging](https://sharanharsoor.github.io/ml-learning-lab/bagging/) to see how trees fitted on different resamples can disagree on the same email. [Return to Decision Trees](https://sharanharsoor.github.io/ml-learning-lab/decision-trees/).